# ATP TOURNAMENTS

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("silver_atp_tournaments").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
tb_atp_matches = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "bronze.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

## Tournaments

In [5]:
df = (
    tb_atp_matches

    .withColumn("TOURNEY_START_DATE", f.min("tourney_date").over(Window.partitionBy("tourney_id")))
    .withColumn("TOURNEY_DRAW_SIZE", f.max("draw_size").over(Window.partitionBy("tourney_id")))

    .select(
        f.col("tourney_id").alias("TOURNEY_ID"),
        f.when(f.col("tourney_name").contains("Davis Cup"), f.lit("Davis Cup"))
            .when(f.col("tourney_id") == '2026-416', f.lit('Rome Masters'))
            .otherwise(f.col("tourney_name")
        ).alias("TOURNEY_NAME"),
        f.when(f.col("tourney_id") == '2026-416', f.lit('M'))
            .when(f.col("tourney_name").contains("Olympics"), f.lit("O"))
            .when(f.col("tourney_name").contains("Finals"), f.lit("F"))
            .otherwise(f.col("tourney_level")
        ).alias("TOURNEY_LEVEL"),
        f.col("TOURNEY_DRAW_SIZE").alias("TOURNEY_DRAW_SIZE").cast('int'),
        f.col("surface").alias("TOURNEY_SURFACE"),
        f.when(f.col("indoor") == 'I', f.lit(True))
            .when(f.col("surface") == 'Carpet', f.lit(True))
            .when(f.col("indoor") == 'O', f.lit(False))
            .when(f.col("tourney_name").contains('Indoor'), f.lit(True)
        ).alias("TOURNEY_IS_INDOOR"),
        f.to_date(f.col("TOURNEY_START_DATE"), 'yyyymmdd').alias("TOURNEY_START_DATE"),
        f.substring(f.col("tourney_date"), 1, 4).alias("REF_YEAR")
    )
    .distinct()
    .orderBy('TOURNEY_START_DATE')
)

## Save dataframe

### Local

In [7]:
df.toPandas().to_csv(
    r"../../data/silver/tb_atp_tournaments.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)